# Sold Mortgage Rate Enrichment

In [1]:
import pandas as pd

sold = pd.read_csv("../../IDX_Data/sold_filtered_week2_3.csv")
print("Shape:", sold.shape)
sold.head()

/var/folders/hx/s9px6scd2pl5dqfp0pldbdgc0000gn/T/ipykernel_21605/3662182009.py:3: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: WaterfrontYN, 3: ListAgentEmail, 4: FireplaceYN, 5: OriginatingSystemName, 6: OriginatingSystemSubName, 7: BuyerAgencyCompensationType, 8: latfilled, 9: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("../../IDX_Data/sold_filtered_week2_3.csv")


Shape: (397172, 84)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,94401,6472.0,NaN,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN
1,SanDiego,SanDiego,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,...,91950,NaN,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN
2,SanDiego,SanDiego,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,...,91950,NaN,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1079166779,davidmartz@compass.com,...,92262,NaN,13504.0,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN
4,Southland,Southland,NaN,False,NaN,NaN,False,1890500.0,1075037759,karen.klein@theagencyre.com,...,91356,0.0,17873.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN


In [2]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(url)
# Rename columns
mortgage.columns = ['date', 'rate_30yr_fixed']
# Convert to datetime
mortgage['date'] = pd.to_datetime(mortgage['date'])

mortgage.head()

,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


In [3]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean()
    .reset_index()
)

mortgage_monthly.head()

,year_month,rate_30yr_fixed
0,1971-04,7.3100
1,1971-05,7.4250
2,1971-06,7.5300
3,1971-07,7.6040
4,1971-08,7.6975


In [4]:
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
sold[['CloseDate', 'year_month']].head()

,CloseDate,year_month
0,2024-01-26,2024-01
1,2024-01-05,2024-01
2,2024-01-05,2024-01
3,2024-01-30,2024-01
4,2024-01-29,2024-01


In [5]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
print("Missing mortgage rates:", sold_with_rates['rate_30yr_fixed'].isnull().sum())

Missing mortgage rates: 0


In [7]:
sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head()

,CloseDate,year_month,ClosePrice,rate_30yr_fixed
0,2024-01-26,2024-01,240000.0,6.6425
1,2024-01-05,2024-01,815000.0,6.6425
2,2024-01-05,2024-01,810000.0,6.6425
3,2024-01-30,2024-01,858000.0,6.6425
4,2024-01-29,2024-01,1890500.0,6.6425


In [8]:
sold_with_rates['rate_30yr_fixed'].describe()

count    397172.000000
mean          6.613009
std           0.294351
min           6.047500
25%           6.352500
50%           6.720000
75%           6.820000
max           7.060000
Name: rate_30yr_fixed, dtype: float64

In [9]:
sold_with_rates.to_csv("sold_with_rates.csv", index=False)
print("Sold dataset with mortgage rates saved.")

Sold dataset with mortgage rates saved.


### Merge Validation

Mortgage rate data was successfully merged into the sold dataset using a monthly key derived from CloseDate.

No missing values were found in the mortgage rate column, confirming a complete and accurate merge.